<a href="https://colab.research.google.com/github/125109486-dev/IS6611-HealthFlow/blob/main/SARIMAX_Backtest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings
warnings.filterwarnings("ignore")

#Configuration
dataFile = "https://raw.githubusercontent.com/125109486-dev/IS6611-HealthFlow/refs/heads/main/synthetic_dataset.csv"
nTestDays = 30
minHistoryRequired = 60
SARMIAX_Order = (1, 1, 1)
SARIMAX_Seasonal_Order = (1, 0, 0, 7)

def load_data(path):
    df = pd.read_csv(path)
    df["date"] = pd.to_datetime(df["date"])
    df["occupancy_rate_pct"] = pd.to_numeric(df["occupancy_rate_pct"], errors="coerce")
    df = df.sort_values("date")
    return df


#Extract a clean, daily-frequency occupancy series for one hospital
def get_hospital_series(df, hospital_name):
    h = df[df["Hospital"] == hospital_name].sort_values("date")
    h = h.drop_duplicates(subset="date").dropna(subset=["occupancy_rate_pct"])
    series = h.set_index("date")["occupancy_rate_pct"].asfreq("D")
    series = series.interpolate(limit_direction="both")
    return series

#Walk-forward backtest for a single hospital. Returns (sarimax_errors, naive_errors) — lists of absolute errors, one per test day.
def backtest_hospital(series, n_test_days, order, seasonal_order):
    sarimax_errors = []
    naive_errors = []

    start = len(series) - n_test_days
    for i in range(start, len(series)):
        train = series.iloc[:i]
        actual = series.iloc[i]

        #Naive baseline: assume tomorrow's occupancy = today's occupancy
        naive_pred = train.iloc[-1]
        naive_errors.append(abs(naive_pred - actual))

        #SARIMAX one-day-ahead forecast, trained only on data before this day
        try:
            model = SARIMAX(
                train, order=order, seasonal_order=seasonal_order,
                enforce_stationarity=False, enforce_invertibility=False
            )
            fit = model.fit(disp=False)
            pred = fit.get_forecast(steps=1).predicted_mean.iloc[0]
            pred = float(np.clip(pred, 0, 100))
            sarimax_errors.append(abs(pred - actual))
        except Exception:

            # If SARIMAX fails to converge on this window, skip this day rather than crashing the whole backtest.
            continue

    return sarimax_errors, naive_errors


def main():
    print(f"Loading {dataFile} ...")
    df = load_data(dataFile)
    hospitals = df["Hospital"].unique()
    print(f"Found {len(hospitals)} hospitals. Running backtest "
          f"(holding out last {nTestDays} days per hospital)...\n")

    per_hospital_results = []
    all_sarimax_errors = []
    all_naive_errors = []

    for hosp in hospitals:
        series = get_hospital_series(df, hosp)

        if len(series) < nTestDays + minHistoryRequired:
            print(f"  Skipping {hosp}: insufficient history ({len(series)} days)")
            continue

        sarimax_errors, naive_errors = backtest_hospital(
            series, nTestDays, SARMIAX_Order, SARIMAX_Seasonal_Order
        )

        if not sarimax_errors:
            print(f"  Skipping {hosp}: SARIMAX failed to converge on all test windows")
            continue

        mae_sarimax = float(np.mean(sarimax_errors))
        mae_naive = float(np.mean(naive_errors))
        improvement_pct = (1 - mae_sarimax / mae_naive) * 100 if mae_naive > 0 else float("nan")

        per_hospital_results.append({
            "hospital": hosp,
            "n_test_days": len(sarimax_errors),
            "mean_occupancy_pct": round(series.mean(), 2),
            "sarimax_mae": round(mae_sarimax, 3),
            "naive_mae": round(mae_naive, 3),
            "improvement_vs_naive_pct": round(improvement_pct, 1),
        })
        all_sarimax_errors.extend(sarimax_errors)
        all_naive_errors.extend(naive_errors)

        print(f"  {hosp:45s}  SARIMAX MAE: {mae_sarimax:.3f}  |  "
              f"Naive MAE: {mae_naive:.3f}  |  Improvement: {improvement_pct:+.1f}%")

    #Summary
    results_df = pd.DataFrame(per_hospital_results)
    results_df.to_csv("backtest_results.csv", index=False)

    overall_sarimax_mae = float(np.mean(all_sarimax_errors))
    overall_naive_mae = float(np.mean(all_naive_errors))
    overall_improvement = (1 - overall_sarimax_mae / overall_naive_mae) * 100
    overall_mean_occ = float(df["occupancy_rate_pct"].mean())
    relative_error_pct = (overall_sarimax_mae / overall_mean_occ) * 100

    summary_lines = [
        "SARIMAX Day-Ahead Occupancy Forecast — Backtest Summary",
        "=" * 58,
        f"Hospitals tested:                {len(results_df)}",
        f"Test days per hospital:          {nTestDays}",
        f"Total forecasts evaluated:        {len(all_sarimax_errors)}",
        "",
        f"SARIMAX mean absolute error:      {overall_sarimax_mae:.3f} percentage points",
        f"Naive baseline mean abs. error:    {overall_naive_mae:.3f} percentage points",
        f"Improvement over naive baseline:   {overall_improvement:.1f}%",
        f"SARIMAX error as % of mean occ.:   {relative_error_pct:.1f}%",
        "",
        "Interpretation:",
        "SARIMAX outperforms a naive 'no change' baseline, indicating the model", "captures genuine predictive signal in the day-ahead occupancy trend.", "Full per-hospital results saved to backtest_results.csv.",
    ]
    summary_text = "\n".join(summary_lines)

    with open("backtest_summary.txt", "w") as f:
        f.write(summary_text)

    print("\n" + summary_text)
    print("\nSaved: backtest_results.csv, backtest_summary.txt")


if __name__ == "__main__":
    main()


Loading https://raw.githubusercontent.com/125109486-dev/IS6611-HealthFlow/refs/heads/main/synthetic_dataset.csv ...
Found 27 hospitals. Running backtest (holding out last 30 days per hospital)...

  Bantry General Hospital                        SARIMAX MAE: 0.917  |  Naive MAE: 1.467  |  Improvement: +37.5%
  Wexford General Hospital                       SARIMAX MAE: 0.116  |  Naive MAE: 0.193  |  Improvement: +40.2%
  University Hospital Limerick                   SARIMAX MAE: 0.991  |  Naive MAE: 1.363  |  Improvement: +27.3%
  University Hospital Kerry                      SARIMAX MAE: 0.296  |  Naive MAE: 0.473  |  Improvement: +37.6%
  Tipperary University Hospital                  SARIMAX MAE: 0.317  |  Naive MAE: 0.537  |  Improvement: +40.9%
  Temple Street Childrens University Hospital    SARIMAX MAE: 0.096  |  Naive MAE: 0.130  |  Improvement: +26.2%
  Tallaght University Hospital                   SARIMAX MAE: 0.116  |  Naive MAE: 0.177  |  Improvement: +34.5%
  St. Vincen